# Full Dataset Multilabel Baseline

This notebook is the full-data training version of `model_fast.ipynb`.

It trains on the manifests produced by `full_dataset_preprocess.ipynb` and improves balance in two ways:

- offline minority clip augmentation from the preprocessing notebook
- online weighted sampling plus class-weighted loss during training

Validation uses only original clips, never augmented variants, and split membership is controlled by `source_video_id` to avoid leakage.


In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.cuda.amp import GradScaler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

warnings.filterwarnings("ignore", category=UserWarning)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def normalize_video_id(value):
    text = str(value).strip()
    if text.endswith(".0"):
        integer_candidate = text[:-2]
        if integer_candidate.isdigit():
            text = integer_candidate
    return text


DATA_ROOT = Path(r"E:\m-hvc\datasets\data-from-juniors")
# DATA_ROOT = Path(r"D:\hvc\datasets")

FRAMES_ROOT = DATA_ROOT / "full_dataset_frames"
# FRAMES_ROOT = DATA_ROOT / "frames"

MANIFEST_PATH = DATA_ROOT / "full_dataset_augmented_manifest.csv"
BASE_MANIFEST_PATH = DATA_ROOT / "full_dataset_base_manifest.csv"
CHECKPOINT_PATH = Path.cwd() /'vision_classification' / "best_full_dataset_model.pt"
CHECKPOINT_PATH_EMA = Path.cwd() / 'vision_classification' / "best_model_fast_baseline_ema.pt"

CFG = {
    "seed": 42,
    "image_size": 224,
    "num_frames": 8,
    "batch_size": 32,
    "num_workers": 4,
    "epochs": 12,
    "warmup_epochs": 1,
    "val_size": 0.20,
    "head_lr": 3e-4,
    "backbone_lr": 3e-5,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "patience": 4,
    "max_pos_weight": 20.0,
    "sampler_weight_cap": 5.0,
}

set_seed(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("device:", DEVICE)
print("manifest:", MANIFEST_PATH)
print("checkpoint:", CHECKPOINT_PATH)


C:\Users\Rasheek\.conda\envs\fyp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
manifest: E:\m-hvc\datasets\data-from-juniors\full_dataset_augmented_manifest.csv
checkpoint: E:\m-hvc\vision_classification\best_full_dataset_model.pt


In [2]:
manifest_df = pd.read_csv(MANIFEST_PATH)
base_manifest_df = pd.read_csv(BASE_MANIFEST_PATH)

manifest_df["source_video_id"] = manifest_df["source_video_id"].map(normalize_video_id)
manifest_df["video_key"] = manifest_df["video_key"].map(lambda value: str(value))
manifest_df["frame_dir"] = manifest_df["frame_dir"].map(lambda value: str(value))
manifest_df["is_augmented"] = manifest_df["is_augmented"].fillna(0).astype(int)

metadata_columns = {
    "video_key",
    "source_video_id",
    "video_path",
    "frame_dir",
    "is_augmented",
    "aug_index",
    "frame_count_extracted",
    "fps",
    "duration_sec",
    "width",
    "height",
}
label_columns = [column for column in manifest_df.columns if column not in metadata_columns]

manifest_df[label_columns] = manifest_df[label_columns].fillna(0).astype(int)
base_manifest_df["source_video_id"] = base_manifest_df["source_video_id"].map(normalize_video_id)
base_manifest_df[label_columns] = base_manifest_df[label_columns].fillna(0).astype(int)

original_df = manifest_df[manifest_df["is_augmented"] == 0].copy().reset_index(drop=True)
augmented_only_df = manifest_df[manifest_df["is_augmented"] == 1].copy().reset_index(drop=True)

print(f"original clips: {len(original_df):,}")
print(f"augmented clips: {len(augmented_only_df):,}")
print(f"total trainable clips in manifest: {len(manifest_df):,}")

label_summary = pd.DataFrame(
    {
        "original_positives": original_df[label_columns].sum().astype(int),
        "augmented_manifest_positives": manifest_df[label_columns].sum().astype(int),
    }
).sort_values("original_positives", ascending=False)

display(label_summary)
original_df.head()


original clips: 5,118
augmented clips: 1,566
total trainable clips in manifest: 6,684


,original_positives,augmented_manifest_positives
humour,4396,5278
sensitive,700,971
anger,373,448
derogatory__lang,351,459
generic,318,424
positive,190,412
emotional,164,334
threat,149,340
political_hate,131,320
gender_hate,109,331


,video_key,source_video_id,video_path,frame_dir,is_augmented,aug_index,frame_count_extracted,fps,duration_sec,width,...,political_hate,religion_hate,informative,ethinity_hate,anger,emotional,social_hate,controversial,indv_hate,gender_hate
0,2,2,E:\m-hvc\datasets\data-from-juniors\videos\2.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
1,9,9,E:\m-hvc\datasets\data-from-juniors\videos\9.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
2,10,10,E:\m-hvc\datasets\data-from-juniors\videos\10.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
3,11,11,E:\m-hvc\datasets\data-from-juniors\videos\11.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,E:\m-hvc\datasets\data-from-juniors\videos\1.mp4,E:\m-hvc\datasets\data-from-juniors\full_datas...,0,0,16,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
def score_split(y_full, y_train, y_val):
    full_prev = y_full.mean(axis=0)
    train_prev = y_train.mean(axis=0)
    val_prev = y_val.mean(axis=0)
    drift = np.abs(train_prev - full_prev).mean() + np.abs(val_prev - full_prev).mean()
    missing_train = int((y_train.sum(axis=0) == 0).sum())
    missing_val = int(((y_full.sum(axis=0) >= 2) & (y_val.sum(axis=0) == 0)).sum())
    return drift + missing_train * 1.0 + missing_val * 0.5


def multilabel_train_val_split(frame_df, label_cols, val_size=0.2, seed=42, tries=200):
    y = frame_df[label_cols].values
    all_indices = np.arange(len(frame_df))

    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=val_size,
            random_state=seed,
        )
        train_idx, val_idx = next(splitter.split(np.zeros(len(frame_df)), y))
        method = "iterstrat"
    except Exception:
        rng = np.random.default_rng(seed)
        best_pair = None
        best_score = float("inf")

        for _ in range(tries):
            random_state = int(rng.integers(0, 1_000_000_000))
            train_idx, val_idx = train_test_split(
                all_indices,
                test_size=val_size,
                random_state=random_state,
                shuffle=True,
            )
            current_score = score_split(y, y[train_idx], y[val_idx])
            if current_score < best_score:
                best_score = current_score
                best_pair = (train_idx, val_idx)

        train_idx, val_idx = best_pair
        method = f"best-of-{tries} random splits"

    train_df = frame_df.iloc[train_idx].reset_index(drop=True)
    val_df = frame_df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df, method


train_base_df, val_df, split_method = multilabel_train_val_split(
    original_df,
    label_columns,
    val_size=CFG["val_size"],
    seed=CFG["seed"],
)

train_source_ids = set(train_base_df["source_video_id"].tolist())
val_source_ids = set(val_df["source_video_id"].tolist())

train_df = manifest_df[manifest_df["source_video_id"].isin(train_source_ids)].copy().reset_index(drop=True)
val_df = original_df[original_df["source_video_id"].isin(val_source_ids)].copy().reset_index(drop=True)

distribution_df = pd.DataFrame(
    {
        "train_original": train_base_df[label_columns].sum().astype(int),
        "train_with_aug": train_df[label_columns].sum().astype(int),
        "val_original": val_df[label_columns].sum().astype(int),
    }
).sort_values("train_original", ascending=False)

print("split method:", split_method)
print(f"train original clips: {len(train_base_df):,}")
print(f"train clips with augmentation: {len(train_df):,}")
print(f"val original clips: {len(val_df):,}")
display(distribution_df)


split method: best-of-200 random splits
train original clips: 4,094
train clips with augmentation: 5,369
val original clips: 1,024


,train_original,train_with_aug,val_original
humour,3513,4229,883
sensitive,561,786,139
anger,304,365,69
derogatory__lang,281,374,70
generic,252,331,66
positive,156,336,34
emotional,134,274,30
threat,119,272,30
political_hate,102,250,29
gender_hate,91,276,18


In [4]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def numeric_frame_key(path: Path):
    stem = path.stem
    return (0, int(stem)) if stem.isdigit() else (1, stem)


def list_frame_paths(frame_dir):
    frame_dir = Path(frame_dir)
    frame_paths = [
        path
        for path in frame_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    return sorted(frame_paths, key=numeric_frame_key)


def sample_frame_paths(frame_dir, num_frames):
    frame_paths = list_frame_paths(frame_dir)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found in {frame_dir}")
    if len(frame_paths) == 1:
        return frame_paths * num_frames
    indices = np.linspace(0, len(frame_paths) - 1, num=num_frames)
    indices = np.clip(np.round(indices).astype(int), 0, len(frame_paths) - 1)
    return [frame_paths[index] for index in indices]


class ClipTransform:
    def __init__(self, image_size=224, train=True):
        self.image_size = image_size
        self.train = train

    def __call__(self, images):
        processed = []

        if self.train:
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                images[0],
                scale=(0.72, 1.0),
                ratio=(0.85, 1.15),
            )
            do_flip = random.random() < 0.5
            brightness = 1.0 + random.uniform(-0.20, 0.20)
            contrast = 1.0 + random.uniform(-0.20, 0.20)
            saturation = 1.0 + random.uniform(-0.20, 0.20)
        else:
            resize_size = int(self.image_size * 1.15)

        for image in images:
            if self.train:
                image = TF.resized_crop(
                    image,
                    i,
                    j,
                    h,
                    w,
                    size=[self.image_size, self.image_size],
                    interpolation=InterpolationMode.BILINEAR,
                )
                if do_flip:
                    image = TF.hflip(image)
                image = TF.adjust_brightness(image, brightness)
                image = TF.adjust_contrast(image, contrast)
                image = TF.adjust_saturation(image, saturation)
            else:
                image = TF.resize(
                    image,
                    size=[resize_size, resize_size],
                    interpolation=InterpolationMode.BILINEAR,
                )
                image = TF.center_crop(image, [self.image_size, self.image_size])

            tensor = TF.to_tensor(image)
            tensor = TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)
            processed.append(tensor)

        return torch.stack(processed, dim=0)


class MultiLabelVideoDataset(Dataset):
    def __init__(self, frame_df, label_cols, transform, num_frames):
        self.frame_df = frame_df.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.frame_df)

    def __getitem__(self, idx):
        row = self.frame_df.iloc[idx]
        frame_paths = sample_frame_paths(row["frame_dir"], self.num_frames)

        images = []
        for frame_path in frame_paths:
            with Image.open(frame_path) as image:
                images.append(image.convert("RGB"))

        clip = self.transform(images)
        label_array = row[self.label_cols].to_numpy(dtype=np.float32, copy=True)
        labels = torch.from_numpy(label_array)
        metadata = {
            "video_key": row["video_key"],
            "source_video_id": row["source_video_id"],
            "is_augmented": int(row["is_augmented"]),
        }
        return clip, labels, metadata


def compute_sample_weights(frame_df, label_cols, max_weight=5.0):
    positive_counts = frame_df[label_cols].sum().clip(lower=1)
    inverse_frequency = 1.0 / positive_counts.values.astype(np.float32)
    weights = []

    for row in frame_df[label_cols].itertuples(index=False):
        positive_vector = np.asarray(row, dtype=np.float32)
        if positive_vector.sum() <= 0:
            sample_weight = float(np.mean(inverse_frequency))
        else:
            sample_weight = float((positive_vector * inverse_frequency).sum() / positive_vector.sum())
        weights.append(sample_weight)

    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.25, max_weight)
    return torch.as_tensor(weights, dtype=torch.double)


train_transform = ClipTransform(image_size=CFG["image_size"], train=True)
val_transform = ClipTransform(image_size=CFG["image_size"], train=False)

train_dataset = MultiLabelVideoDataset(train_df, label_columns, train_transform, CFG["num_frames"])
val_dataset = MultiLabelVideoDataset(val_df, label_columns, val_transform, CFG["num_frames"])

train_weights = compute_sample_weights(train_df, label_columns, max_weight=CFG["sampler_weight_cap"])
train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    sampler=train_sampler,
    # num_workers=CFG["num_workers"],
    pin_memory=USE_AMP,
    drop_last=False,
    # persistent_workers=CFG["num_workers"] > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    # num_workers=CFG["num_workers"],
    pin_memory=USE_AMP,
    drop_last=False,
    # persistent_workers=CFG["num_workers"] > 0,
)

weighted_distribution = pd.DataFrame(
    {
        "train_original": train_base_df[label_columns].sum().astype(int),
        "train_with_aug": train_df[label_columns].sum().astype(int),
        "weighted_sampler_mass": np.dot(train_df[label_columns].T.values, train_weights.numpy()).round(2),
    },
    index=label_columns,
).sort_values("weighted_sampler_mass", ascending=False)

display(weighted_distribution)

sample_clip, sample_targets, sample_meta = train_dataset[0]
print("sample video:", sample_meta)
print("sample clip shape:", tuple(sample_clip.shape))
print(
    "positive labels:",
    [label_columns[index] for index, value in enumerate(sample_targets.tolist()) if value > 0],
)


,train_original,train_with_aug,weighted_sampler_mass
humour,3513,4229,2603.58
sensitive,561,786,878.08
generic,252,331,543.46
derogatory__lang,281,374,481.07
political_hate,102,250,476.92
positive,156,336,465.59
threat,119,272,459.94
informative,48,192,439.51
anger,304,365,423.80
indv_hate,55,220,417.33


sample video: {'video_key': '2', 'source_video_id': '2', 'is_augmented': 0}
sample clip shape: (8, 3, 224, 224)
positive labels: ['generic']


In [5]:
def load_efficientnet_v2_s():
    if not hasattr(tv_models, "efficientnet_v2_s"):
        raise AttributeError("efficientnet_v2_s is not available in this torchvision build")

    builder = tv_models.efficientnet_v2_s
    weights_enum = getattr(tv_models, "EfficientNet_V2_S_Weights", None)

    try:
        if weights_enum is not None:
            model = builder(weights=weights_enum.DEFAULT)
        else:
            model = builder(pretrained=True)
        pretrained = True
    except Exception as exc:
        print(f"EfficientNetV2-S pretrained weights unavailable: {exc}")
        try:
            model = builder(weights=None)
        except TypeError:
            model = builder(pretrained=False)
        pretrained = False

    feature_dim = model.classifier[1].in_features
    model.classifier = nn.Identity()
    return model, feature_dim, pretrained


def load_resnet50():
    builder = tv_models.resnet50
    weights_enum = getattr(tv_models, "ResNet50_Weights", None)

    try:
        if weights_enum is not None:
            model = builder(weights=weights_enum.DEFAULT)
        else:
            model = builder(pretrained=True)
        pretrained = True
    except Exception as exc:
        print(f"ResNet50 pretrained weights unavailable: {exc}")
        try:
            model = builder(weights=None)
        except TypeError:
            model = builder(pretrained=False)
        pretrained = False

    feature_dim = model.fc.in_features
    model.fc = nn.Identity()
    return model, feature_dim, pretrained


def build_backbone(preferred=None):
    candidates = [preferred] if preferred else ["efficientnet_v2_s", "resnet50"]

    for backbone_name in candidates:
        if backbone_name == "efficientnet_v2_s":
            try:
                backbone, feature_dim, pretrained = load_efficientnet_v2_s()
                return backbone_name, backbone, feature_dim, pretrained
            except Exception as exc:
                print(f"Skipping efficientnet_v2_s: {exc}")
        elif backbone_name == "resnet50":
            backbone, feature_dim, pretrained = load_resnet50()
            return backbone_name, backbone, feature_dim, pretrained
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

    raise RuntimeError("Could not build a supported backbone")


class VideoBaseline(nn.Module):
    def __init__(self, num_labels, backbone_name=None):
        super().__init__()
        self.backbone_name, self.backbone, feature_dim, self.pretrained = build_backbone(backbone_name)
        self.temporal_attention = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(0.35),
            nn.Linear(feature_dim, num_labels),
        )

    def forward(self, clips):
        batch_size, num_frames, channels, height, width = clips.shape
        x = clips.view(batch_size * num_frames, channels, height, width)
        features = self.backbone(x)
        features = features.view(batch_size, num_frames, -1)
        attention = torch.softmax(self.temporal_attention(features).squeeze(-1), dim=1)
        pooled = (features * attention.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled)


def freeze_backbone_batchnorm(model):
    for module in model.backbone.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            for parameter in module.parameters():
                parameter.requires_grad = False


def set_backbone_trainable(model, trainable):
    for parameter in model.backbone.parameters():
        parameter.requires_grad = trainable
    freeze_backbone_batchnorm(model)


def build_pos_weight(frame_df, label_cols, max_pos_weight):
    positives = frame_df[label_cols].sum().clip(lower=1)
    negatives = len(frame_df) - positives
    weights = (negatives / positives).clip(lower=1.0, upper=max_pos_weight)
    return torch.tensor(weights.values, dtype=torch.float32)

def build_pos_weight(frame_df, label_cols, max_pos_weight):
    positives = frame_df[label_cols].sum().clip(lower=1)
    negatives = len(frame_df) - positives
    weights = (negatives / positives).clip(lower=1.0, upper=max_pos_weight)
    return torch.tensor(weights.values, dtype=torch.float32)


# model = VideoBaseline(len(label_columns)).to(DEVICE)
# set_backbone_trainable(model, False)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models


# ---------------------------
# Backbone loaders (unchanged)
# ---------------------------

def load_efficientnet_v2_s():
    builder = tv_models.efficientnet_v2_s
    weights_enum = getattr(tv_models, "EfficientNet_V2_S_Weights", None)

    try:
        model = builder(weights=weights_enum.DEFAULT) if weights_enum else builder(pretrained=True)
        pretrained = True
    except Exception:
        model = builder(weights=None) if weights_enum else builder(pretrained=False)
        pretrained = False

    feature_dim = model.classifier[1].in_features
    model.classifier = nn.Identity()
    return model, feature_dim, pretrained


def load_resnet50():
    builder = tv_models.resnet50
    weights_enum = getattr(tv_models, "ResNet50_Weights", None)

    try:
        model = builder(weights=weights_enum.DEFAULT) if weights_enum else builder(pretrained=True)
        pretrained = True
    except Exception:
        model = builder(weights=None) if weights_enum else builder(pretrained=False)
        pretrained = False

    feature_dim = model.fc.in_features
    model.fc = nn.Identity()
    return model, feature_dim, pretrained


def build_backbone(name=None):
    if name == "resnet50":
        return "resnet50", *load_resnet50()
    return "efficientnet_v2_s", *load_efficientnet_v2_s()


# ---------------------------
# Strong Video Model
# ---------------------------

class VideoModel(nn.Module):
    def __init__(self, num_labels, backbone_name=None):
        super().__init__()

        self.backbone_name, self.backbone, feature_dim, self.pretrained = build_backbone(backbone_name)

        # ---- Feature projection (critical) ----
        self.embed_dim = 512
        self.feature_proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, self.embed_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, 500, self.embed_dim))
        

        # ---- CLS token ----
        self.cls_token = nn.Parameter(torch.randn(1, 1, self.embed_dim))

        # ---- Transformer temporal modeling ----
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=8,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # ---- Classification head ----
        self.head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels),
        )

    def forward(self, clips):
        """
        clips: (B, T, C, H, W)
        """
        B, T, C, H, W = clips.shape

        # ---- frame-wise feature extraction ----
        x = clips.view(B * T, C, H, W)
        features = self.backbone(x)                # (B*T, F)
        features = features.view(B, T, -1)         # (B, T, F)

        # ---- project features ----
        features = self.feature_proj(features)     # (B, T, 512)

        # ---- add CLS token ----
        cls_tokens = self.cls_token.expand(B, -1, -1)
        features = torch.cat([cls_tokens, features], dim=1)  # (B, T+1, 512)
        # ---- temporal modeling ----
        features = features + self.pos_embed[:, :features.size(1)]
        features = self.temporal_encoder(features)

        # ---- classification using CLS ----
        pooled = features[:, 0]

        return self.head(pooled)


# ---------------------------
# Training utilities
# ---------------------------

def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable


def unfreeze_last_layers(model, n=2):
    children = list(model.backbone.children())
    for layer in children[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


# ---------------------------
# Optional: Focal Loss (better than BCE)
# ---------------------------

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_factor = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        modulating_factor = (1 - p_t) ** self.gamma

        loss = alpha_factor * modulating_factor * ce_loss
        return loss.mean()
    
    
model = VideoModel(len(label_columns)).to(DEVICE)
set_backbone_trainable(model, False)
unfreeze_last_layers(model, 2)

pos_weight = build_pos_weight(train_df, label_columns, CFG["max_pos_weight"]).to(DEVICE)
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion = FocalLoss()
scaler = GradScaler(enabled=USE_AMP)

print("backbone:", model.backbone_name, "| pretrained:", model.pretrained)
print(
    "trainable parameters:",
    f"{sum(param.numel() for param in model.parameters() if param.requires_grad):,}",
)
pd.Series(pos_weight.detach().cpu().numpy(), index=label_columns).sort_values(
    ascending=False
)




# model = VideoBaseline(len(label_columns)).to(DEVICE)
# set_backbone_trainable(model, False)

# pos_weight = build_pos_weight(train_base_df, label_columns, CFG["max_pos_weight"]).to(DEVICE)
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# scaler = GradScaler(enabled=USE_AMP)

# print("backbone:", model.backbone_name, "| pretrained:", model.pretrained)
# print(
#     "trainable parameters:",
#     f"{sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad):,}",
# )
# pd.Series(pos_weight.detach().cpu().numpy(), index=label_columns).sort_values(ascending=False)


backbone: efficientnet_v2_s | pretrained: True
trainable parameters: 5,257,747


C:\Users\Rasheek\AppData\Local\Temp\ipykernel_25812\588546760.py:289: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)


political_hate      20.000000
ethinity_hate       20.000000
indv_hate           20.000000
controversial       20.000000
social_hate         20.000000
sexuality_hate      20.000000
nationality_hate    20.000000
caste_based_hate    20.000000
religion_hate       20.000000
informative         20.000000
threat              18.738970
emotional           18.594891
gender_hate         18.452898
generic             15.220544
positive            14.979167
anger               13.709589
derogatory__lang    13.355615
sensitive            5.830789
humour               1.000000
dtype: float32

In [6]:
# ---- EMA ----
from copy import deepcopy

ema_decay = 0.999

def update_ema(model, ema_model):
    with torch.no_grad():
        for p, ema_p in zip(model.parameters(), ema_model.parameters()):
            ema_p.data.mul_(ema_decay).add_(p.data, alpha=1 - ema_decay)


In [7]:
def build_optimizer(model, backbone_lr, head_lr):
    backbone_params = [
        p for p in model.backbone.parameters() if p.requires_grad
    ]

    other_params = []

    # feature projection
    other_params += list(model.feature_proj.parameters())

    # transformer
    other_params += list(model.temporal_encoder.parameters())

    # head
    other_params += list(model.head.parameters())

    # CLS token (important)
    other_params.append(model.cls_token)

    other_params = [p for p in other_params if p.requires_grad]

    param_groups = []

    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": backbone_lr})

    if other_params:
        param_groups.append({"params": other_params, "lr": head_lr})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    return optimizer


def optimize_thresholds(y_true, y_prob, default=0.5):
    grid = np.linspace(0.15, 0.85, 15)
    thresholds = []

    for label_index in range(y_true.shape[1]):
        if y_true[:, label_index].sum() == 0:
            thresholds.append(default)
            continue

        best_threshold = default
        best_score = -1.0

        for threshold in grid:
            preds = (y_prob[:, label_index] >= threshold).astype(np.int32)
            score = f1_score(y_true[:, label_index], preds, zero_division=0)
            if score > best_score:
                best_score = score
                best_threshold = float(threshold)

        thresholds.append(best_threshold)

    return np.array(thresholds, dtype=np.float32)


def compute_metrics(y_true, y_prob, thresholds):
    thresholds = np.asarray(thresholds, dtype=np.float32)
    if not np.isfinite(y_prob).all():
        sample_index, label_index = np.argwhere(~np.isfinite(y_prob))[0]
        label_name = label_columns[label_index] if "label_columns" in globals() else label_index
        raise ValueError(
            f"y_prob contains non-finite values at sample {sample_index}, label {label_name}"
        )

    preds = (y_prob >= thresholds.reshape(1, -1)).astype(np.int32)

    per_label_f1 = f1_score(y_true, preds, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, preds, average="micro", zero_division=0)
    hamming_acc = (preds == y_true).mean()
    subset_acc = (preds == y_true).all(axis=1).mean()
    label_acc = hamming_acc
    exact_match_acc = subset_acc

    ap_scores = []
    for label_index in range(y_true.shape[1]):
        if len(np.unique(y_true[:, label_index])) < 2:
            ap_scores.append(np.nan)
        else:
            ap_scores.append(
                average_precision_score(y_true[:, label_index], y_prob[:, label_index])
            )

    ap_scores = np.asarray(ap_scores, dtype=np.float32)
    macro_ap = float(np.nanmean(ap_scores)) if not np.isnan(ap_scores).all() else float("nan")

    return {
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "macro_ap": macro_ap,
        "hamming_acc": float(hamming_acc),
        "subset_acc": float(subset_acc),
        "label_acc": float(label_acc),
        "exact_match_acc": float(exact_match_acc),
        "per_label_f1": per_label_f1,
        "preds": preds,
    }


def ensure_finite_tensor(name, tensor, video_ids=None):
    if torch.isfinite(tensor).all():
        return

    if tensor.ndim == 0:
        message = f"{name} became non-finite"
    else:
        bad_index = tuple(torch.nonzero(~torch.isfinite(tensor), as_tuple=False)[0].tolist())
        message = f"{name} contains non-finite values at index {bad_index}"
        if video_ids is not None and len(video_ids) > 0:
            message += f" | sample={video_ids[bad_index[0]]}"

    raise RuntimeError(message)


def run_epoch(model, loader, criterion, optimizer=None, scaler=None,ema_model=None):
    is_train = optimizer is not None
    model.train(is_train)
    if is_train:
        freeze_backbone_batchnorm(model)

    total_loss = 0.0
    all_probs = []
    all_targets = []

    progress = tqdm(loader, leave=False)
    progress.set_description("train" if is_train else "val")

    grad_context = torch.enable_grad() if is_train else torch.no_grad()
    amp_enabled = bool(scaler is not None and scaler.is_enabled())

    with grad_context:
        for clips, labels, video_ids in progress:
            clips = clips.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=DEVICE.type, enabled=amp_enabled):
                logits = model(clips)
                loss = criterion(logits, labels)
            ensure_finite_tensor("logits", logits, video_ids)
            ensure_finite_tensor("loss", loss, video_ids)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                
            if ema_model is not None:
                update_ema(model, ema_model)
            

            probs = torch.sigmoid(logits)
            ensure_finite_tensor("probabilities", probs, video_ids)
            total_loss += loss.item() * clips.size(0)
            all_probs.append(probs.detach().cpu())
            all_targets.append(labels.detach().cpu())
            progress.set_postfix(loss=f"{loss.item():.4f}")

    y_prob = torch.cat(all_probs).numpy()
    y_true = torch.cat(all_targets).numpy().astype(np.int32)
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, y_true, y_prob


def save_checkpoint(path, model, thresholds, history):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "label_columns": label_columns,
            "config": CFG,
            "thresholds": thresholds,
            "history": history,
            "backbone": model.backbone_name,
        },
        path,
    )


In [8]:
optimizer = build_optimizer(model, CFG["backbone_lr"], CFG["head_lr"])
scheduler = None
ema_model = deepcopy(model)

best_macro_f1 = -1.0
best_thresholds = np.full(len(label_columns), 0.5, dtype=np.float32)
best_val_y = None
best_val_prob = None
history = []
epochs_without_improvement = 0

for epoch in range(1, CFG["epochs"] + 1):
    if epoch == CFG["warmup_epochs"] + 1:
        set_backbone_trainable(model, True)
        optimizer = build_optimizer(model, CFG["backbone_lr"], CFG["head_lr"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, CFG["epochs"] - CFG["warmup_epochs"]),
        )
        print(f"epoch {epoch:02d}: backbone unfrozen")

    train_loss, train_y, train_prob = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        scaler=scaler,
        ema_model=ema_model
    )
    val_loss, val_y, val_prob = run_epoch(model, val_loader, criterion,ema_model=ema_model)

    thresholds = optimize_thresholds(val_y, val_prob, default=0.5)
    train_metrics = compute_metrics(train_y, train_prob, thresholds)
    val_metrics = compute_metrics(val_y, val_prob, thresholds)

    if scheduler is not None:
        scheduler.step()

    epoch_row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_macro_f1": train_metrics["macro_f1"],
        "val_macro_f1": val_metrics["macro_f1"],
        "train_label_acc": train_metrics["label_acc"],
        "val_label_acc": val_metrics["label_acc"],
        "train_exact_match_acc": train_metrics["exact_match_acc"],
        "val_exact_match_acc": val_metrics["exact_match_acc"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_ap": val_metrics["macro_ap"],
    }
    history.append(epoch_row)

    print(
        f"epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_acc={train_metrics['label_acc']:.4f} | "
        f"val_acc={val_metrics['label_acc']:.4f} | "
        f"train_exact_acc={train_metrics['exact_match_acc']:.4f} | "
        f"val_exact_acc={val_metrics['exact_match_acc']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
        f"val_micro_f1={val_metrics['micro_f1']:.4f} | "
        f"val_macro_ap={val_metrics['macro_ap']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_thresholds = thresholds.copy()
        best_val_y = val_y.copy()
        best_val_prob = val_prob.copy()
        epochs_without_improvement = 0
        save_checkpoint(CHECKPOINT_PATH, model, best_thresholds, history)
        save_checkpoint(CHECKPOINT_PATH_EMA, ema_model, best_thresholds, history)
        
        print(f"saved best checkpoint -> {CHECKPOINT_PATH.name}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG["patience"]:
            print("early stopping triggered")
            break

history_df = pd.DataFrame(history)
display(history_df)

best_val_metrics = compute_metrics(best_val_y, best_val_prob, best_thresholds)
per_label_df = pd.DataFrame(
    {
        "label": label_columns,
        "threshold": best_thresholds,
        "val_f1": best_val_metrics["per_label_f1"],
        "train_original_positives": train_base_df[label_columns].sum().values,
        "train_with_aug_positives": train_df[label_columns].sum().values,
        "val_positives": val_df[label_columns].sum().values,
    }
).sort_values("val_f1", ascending=False)

display(per_label_df)


epoch 01 | train_loss=0.0288 | val_loss=0.0183 | train_acc=0.7099 | val_acc=0.8440 | train_exact_acc=0.0015 | val_exact_acc=0.0254 | val_macro_f1=0.1652 | val_micro_f1=0.4268 | val_macro_ap=0.1228


RuntimeError: Parent directory E:\m-hvc\vision_classification does not exist.

In [ ]:
def load_checkpoint(checkpoint_path=CHECKPOINT_PATH, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = VideoBaseline(
        len(checkpoint["label_columns"]),
        backbone_name=checkpoint.get("backbone"),
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model, checkpoint


def predict_from_frame_dir(frame_dir, checkpoint_path=CHECKPOINT_PATH):
    model, checkpoint = load_checkpoint(checkpoint_path)
    frame_dir = Path(frame_dir)
    frame_paths = sample_frame_paths(frame_dir, checkpoint["config"]["num_frames"])

    images = []
    for frame_path in frame_paths:
        with Image.open(frame_path) as image:
            images.append(image.convert("RGB"))

    inference_transform = ClipTransform(
        image_size=checkpoint["config"]["image_size"],
        train=False,
    )
    clip = inference_transform(images).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.sigmoid(model(clip)).squeeze(0).cpu().numpy()

    thresholds = np.asarray(checkpoint["thresholds"], dtype=np.float32)
    preds = (probs >= thresholds).astype(int)

    return pd.DataFrame(
        {
            "label": checkpoint["label_columns"],
            "prob": probs,
            "threshold": thresholds,
            "pred": preds,
        }
    ).sort_values("prob", ascending=False)


# Example usage after training:
# predict_from_frame_dir(FRAMES_ROOT / str(train_base_df.loc[0, "source_video_id"]))
